# ☀️ HYPERION: Supreme Tactical Intelligence
**Kaggle Competition: Orbit Wars | Public Score: 700+ (Top Tier)**

HYPERION completely rewrites the tactical metagame of Orbit Wars. Evolving from the OMEGA series, HYPERION shifts from purely reactive algorithms to proactive, highly aggressive early-game dominance and multi-directional combat.

## 🚀 The HYPERION Innovations:
1. **Opening Blitz (Turns 1-22):** Halves reserves to mathematically snowball production faster than any opponent.
2. **Flanking Multiplier (×1.20):** Overloads enemy decision-making by coordinating strikes from 2+ planets simultaneously.
3. **Tsunami Decision Tree:** A 4-level math-driven protocol (Full, Cheap, Soft, Standard) to optimize fleet velocities and claim free production ships.
4. **Total War Vectoring:** Late-game targeting is prioritized not just by distance, but by `production / distance`.

## 1. Opening Blitz (The Power of Compound Production)
The first 22 turns of the game decide the victor. Capturing a `production=3` planet at Turn 5 yields **1485 ships** over the game. Capturing it at Turn 50 yields only **1350 ships**.

HYPERION activates an "Opening Blitz" for the first 22 turns. It overrides standard defensive policy, reducing ship reserves by 45% (`OPENING_BLITZ_RESERVE_FRAC = 0.55`) and actively hunting down high-value neutral planets with a `×1.28` boost multiplier. The compound advantage of an early economy completely negates the risk.

In [ ]:
# --- EXCERPT: OPENING BLITZ PROTOCOL ---
OPENING_BLITZ_TURNS        = 22
OPENING_BLITZ_RESERVE_FRAC = 0.55
OPENING_BLITZ_MARGIN_FRAC  = 0.70
OPENING_BLITZ_NEUTRAL_VM   = 1.28

# Inside Policy Builder
if world.in_blitz:
    exact  = int(exact  * OPENING_BLITZ_RESERVE_FRAC)
    proact = int(proact * OPENING_BLITZ_RESERVE_FRAC)
    
# Inside Target Scoring
if target.owner == -1 and world.in_blitz:                   
    val *= OPENING_BLITZ_NEUTRAL_VM

## 2. Flanking Multiplier (Multi-Directional Warfare)
When 2 or more of our planets can reach the same enemy target within a `FLANKING_ETA_TOL=3` turn window, the enemy cannot optimally defend. They must either split their forces (making both weak) or abandon one front.

HYPERION dynamically creates a `flankable_ids` map every turn. Any enemy planet that can be flanked receives a pure `×1.20` value multiplier, making it a prime target for swarms.

In [ ]:
# --- EXCERPT: FLANKING MAP CALCULATION ---
def compute_flanking_map(my_planets, planets, init_by_id, ang_vel, comets, comet_ids):
    flankable = set()
    targets = [p for p in planets if p.owner not in (-1,)]  
    
    for target in targets:
        etas = []
        for src in my_planets:
            aim = aim_with_prediction(src, target, max(1, int(src.ships // 2)),
                                      init_by_id, ang_vel, comets, comet_ids)
            if aim is None: continue
            etas.append(aim[1])
            
        if len(etas) >= 2:
            etas.sort()
            for i in range(len(etas) - 1):
                # If two attacks land within 3 turns, it's a flank
                if etas[i+1] - etas[i] <= FLANKING_ETA_TOL:
                    flankable.add(target.id)
                    break
    return flankable

## 3. Tsunami Decision Tree
Orbit Wars fleet speed is logarithmic: `speed(n) = 1.0 + 5.0 × (ln(n) / ln(1000))^1.5`.

If you need 60 ships to capture a planet 35 units away, sending exactly 60 ships takes **11 turns**. If you send 150 ships, it takes **9 turns**. If the planet produces 3 ships per turn, saving 2 turns means you gain **6 FREE ships**.

HYPERION calculates if the "free ships" justify sending a larger fleet. If yes, it unleashes a **FULL TSUNAMI** (90% of budget).

In [ ]:
# --- EXCERPT: TSUNAMI DECISION TREE ---
def speed_optimal_send(needed, available, distance, prod_per_turn):
    if available <= 0 or needed <= 0: return max(1, needed)
    if available <= needed: return needed
    
    base_speed = fleet_speed(max(1, needed))
    base_turns = max(1, int(math.ceil(distance / base_speed)))
    
    if available >= needed * TSUNAMI_THRESH and available >= TSUNAMI_MIN_SHIPS:
        candidate  = min(available, max(needed, int(available * TSUNAMI_RATIO)))
        cand_speed = fleet_speed(max(1, candidate))
        cand_turns = max(1, int(math.ceil(distance / cand_speed)))
        
        turns_saved = base_turns - cand_turns
        extra_ships = candidate - needed
        
        # 1. FULL TSUNAMI: Does it save enough turns to matter?
        if turns_saved >= TSUNAMI_TURNS_SAVED_MIN and prod_per_turn > 0:
            return candidate
            
        # 2. CHEAP TSUNAMI: Are the extra ships cheap enough to send anyway?
        if extra_ships <= available * TSUNAMI_MAX_EXTRA_FRAC:
            return candidate
            
    # 3. SOFT TSUNAMI: A modest 22% increase to see if it saves 1 turn
    modest = min(available, int(needed * 1.22))
    if modest > needed:
        mod_turns = max(1, int(math.ceil(distance / fleet_speed(max(1, modest)))))
        if base_turns - mod_turns >= 1:
            return modest
            
    # 4. STANDARD: Just send what's needed + 6% margin
    return min(available, max(needed, int(needed * 1.06)))